# Embedding Analysis on MVTec AD

This notebook explores learned representations from:
- autoencoders
- PatchCore feature extractors

## Goals
- extract embeddings from train/test images
- visualize them using PCA and t-SNE
- compare normal vs anomalous samples
- better understand why some methods perform better than others

## Imports

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import random
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

from datasets.mvtec import MvtecAdDataset
from models.autoencoder import AutoEncoder
from models.autoencoder_v2 import AutoEncoderV2
from models.patchcore import (
    FeatureExtractor,
    MultiScaleFeatureExtractor,
    extract_patch_embeddings,
)
from src.config import DATA_DIR, BATCH_SIZE
from utils.normalization import preprocess_image

## Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Experiment Configuration

In [ ]:
CATEGORY = "capsule"

ANALYSIS_MODE = "patchcore"  # "autoencoder" or "patchcore"
MODEL_VERSION = "v2"  # used only for autoencoder
EXTRACTOR_TYPE = "multiscale"  # "layer3" or "multiscale", used for patchcore

CHECKPOINT_PATH = "../models/checkpoints/20260406_173424_capsule_autoencoder_v2.pth"  # autoencoder checkpoint if needed

USE_TSNE = True
USE_PCA = True

MAX_SAMPLES = 2000  # important for t-SNE
RANDOM_SEED = 42

OUTPUT_DIR = Path("outputs")
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{CATEGORY}_{ANALYSIS_MODE}"
print("Run:", RUN_NAME)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## Data Loading

In [ ]:
train_dataset = MvtecAdDataset(
    root_dir=DATA_DIR,
    category=CATEGORY,
    split="train",
    transform=preprocess_image,
)

test_dataset = MvtecAdDataset(
    root_dir=DATA_DIR,
    category=CATEGORY,
    split="test",
    transform=preprocess_image,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

## Partie A — Autoencoder embeddings

## Autoencoder Embeddings
Use this section if `ANALYSIS_MODE = "autoencoder"`.

In [ ]:
# Model loading
if ANALYSIS_MODE == "autoencoder":
    if MODEL_VERSION == "v1":
        model = AutoEncoder().to(DEVICE)
    elif MODEL_VERSION == "v2":
        model = AutoEncoderV2().to(DEVICE)
    else:
        raise ValueError("MODEL_VERSION must be 'v1' or 'v2'")

    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    model.eval()

    print("Loaded autoencoder:", model.__class__.__name__)

### Embedding extraction function

In [ ]:
def extract_autoencoder_embeddings(
    model: torch.nn.Module,
    dataloader: DataLoader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract bottleneck embeddings and labels from an autoencoder.

    Returns:
        embeddings: shape (N, D)
        labels: shape (N,)
    """
    embeddings = []
    labels = []

    model.eval()

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            batch_labels = batch["label"]

            z = model.encoder(images)
            z = z.view(z.size(0), -1)

            embeddings.append(z.cpu().numpy())
            labels.append(batch_labels.numpy())

    embeddings = np.concatenate(embeddings, axis=0)
    labels = np.concatenate(labels, axis=0)

    return embeddings, labels

## Partie B — PatchCore embeddings

### PatchCore Embeddings
Use this section if `ANALYSIS_MODE = "patchcore"`.

### Extractor loading

In [ ]:
if ANALYSIS_MODE == "patchcore":
    if EXTRACTOR_TYPE == "layer3":
        extractor = FeatureExtractor().to(DEVICE)
    elif EXTRACTOR_TYPE == "multiscale":
        extractor = MultiScaleFeatureExtractor().to(DEVICE)
    else:
        raise ValueError("EXTRACTOR_TYPE must be 'layer3' or 'multiscale'")

    extractor.eval()
    print("Loaded extractor:", extractor.__class__.__name__)

### Patch embedding extraction function

In [ ]:
def extract_patchcore_embeddings(
    feature_extractor: torch.nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    max_samples: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract patch embeddings and patch-level labels from a dataset.

    For test images:
        all patches inherit the image-level label (rough approximation).

    Returns:
        embeddings: shape (N, D)
        labels: shape (N,)
    """
    all_embeddings = []
    all_labels = []

    feature_extractor.eval()

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            batch_labels = batch["label"]

            feature_map = feature_extractor(images)
            patches = extract_patch_embeddings(feature_map).cpu().numpy()

            batch_size, _, h, w = feature_map.shape
            repeated_labels = np.repeat(batch_labels.numpy(), h * w)

            all_embeddings.append(patches)
            all_labels.append(repeated_labels)

    embeddings = np.concatenate(all_embeddings, axis=0)
    labels = np.concatenate(all_labels, axis=0)

    if embeddings.shape[0] > max_samples:
        indices = np.random.choice(embeddings.shape[0], size=max_samples, replace=False)
        embeddings = embeddings[indices]
        labels = labels[indices]

    return embeddings, labels

In [ ]:
import numpy as np
import torch

from models.patchcore import compute_patchwise_distances, extract_patch_embeddings


def extract_patchcore_embeddings_with_scores(
    feature_extractor: torch.nn.Module,
    dataloader: DataLoader,
    memory_bank: torch.Tensor,
    device: torch.device,
    max_samples: int = 2000,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract patch embeddings, patch-level labels, and PatchCore anomaly scores.

    For anomalous images, all patches inherit the image-level label (approximation).

    Args:
        feature_extractor (torch.nn.Module): PatchCore feature extractor.
        dataloader (DataLoader): DataLoader.
        memory_bank (torch.Tensor): Memory bank of shape (N_train, C).
        device (torch.device): Device.
        max_samples (int, optional): Maximum number of sampled patches.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]:
            - embeddings: shape (N, D)
            - labels: shape (N,)
            - scores: shape (N,)
    """
    all_embeddings = []
    all_labels = []
    all_scores = []

    feature_extractor.eval()
    memory_bank = memory_bank.to(device)

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            batch_labels = batch["label"]

            feature_map = feature_extractor(images)
            patches = extract_patch_embeddings(feature_map)

            patch_scores = compute_patchwise_distances(
                test_patches=patches,
                memory_bank=memory_bank,
            )

            batch_size, _, h, w = feature_map.shape
            repeated_labels = np.repeat(batch_labels.cpu().numpy(), h * w)

            all_embeddings.append(patches.cpu().numpy())
            all_labels.append(repeated_labels)
            all_scores.append(patch_scores.cpu().numpy())

    embeddings = np.concatenate(all_embeddings, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    scores = np.concatenate(all_scores, axis=0)

    if embeddings.shape[0] > max_samples:
        indices = np.random.choice(embeddings.shape[0], size=max_samples, replace=False)
        embeddings = embeddings[indices]
        labels = labels[indices]
        scores = scores[indices]

    return embeddings, labels, scores

### Extraction

In [ ]:
MEMORY_BANK_PATH = (
    "../outputs/memory_banks/20260418_134126_capsule_patchcore_multiscale.pt"
)

saved = torch.load(MEMORY_BANK_PATH, map_location="cpu")
memory_bank = saved["memory_bank"]

print("Loaded memory bank shape:", memory_bank.shape)
print("Category:", saved.get("category", "unknown"))
print("Extractor type:", saved.get("extractor_type", "unknown"))

In [ ]:
if ANALYSIS_MODE == "autoencoder":
    train_embeddings, train_labels = extract_autoencoder_embeddings(
        model, train_loader, DEVICE
    )
    test_embeddings, test_labels = extract_autoencoder_embeddings(
        model, test_loader, DEVICE
    )

elif ANALYSIS_MODE == "patchcore":
    train_embeddings, train_labels, train_scores = (
        extract_patchcore_embeddings_with_scores(
            extractor, train_loader, memory_bank, DEVICE, max_samples=MAX_SAMPLES
        )
    )

    test_embeddings, test_labels, test_scores = (
        extract_patchcore_embeddings_with_scores(
            extractor, test_loader, memory_bank, DEVICE, max_samples=MAX_SAMPLES
        )
    )

print("Train embeddings:", train_embeddings.shape)
print("Test embeddings:", test_embeddings.shape)

## Merge train + test

In [ ]:
X = np.concatenate([train_embeddings, test_embeddings], axis=0)
y = np.concatenate([train_labels, test_labels], axis=0)
scores = np.concatenate([train_scores, test_scores], axis=0)
split = np.array(["train"] * len(train_embeddings) + ["test"] * len(test_embeddings))

## PCA

In [ ]:
if USE_PCA:
    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    X_pca = pca.fit_transform(X)

    print("Explained variance ratio:", pca.explained_variance_ratio_)

### PCA plot by label

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_pca[y == 0, 0],
    X_pca[y == 0, 1],
    s=8,
    alpha=0.5,
    label="Normal",
)

plt.scatter(
    X_pca[y == 1, 0],
    X_pca[y == 1, 1],
    s=8,
    alpha=0.5,
    label="Anomaly",
)

plt.title(f"PCA - {CATEGORY} - {ANALYSIS_MODE}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True)
plt.show()

## t-SNE

In [ ]:
if USE_TSNE:
    tsne = TSNE(
        n_components=2,
        perplexity=30,
        random_state=RANDOM_SEED,
        init="pca",
        learning_rate="auto",
    )
    X_tsne = tsne.fit_transform(X)

### t-SNE plot by label

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_tsne[y == 0, 0],
    X_tsne[y == 0, 1],
    s=8,
    alpha=0.5,
    label="Normal",
)

plt.scatter(
    X_tsne[y == 1, 0],
    X_tsne[y == 1, 1],
    s=8,
    alpha=0.5,
    label="Anomaly",
)

plt.title(f"t-SNE - {CATEGORY} - {ANALYSIS_MODE}")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    scores[y == 0],
    bins=50,
    alpha=0.5,
    label="Normal",
)

plt.hist(
    scores[y == 1],
    bins=50,
    alpha=0.5,
    label="Anomaly",
)

plt.title(f"Score Distribution - {CATEGORY} - {ANALYSIS_MODE}")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.show()